# Classificação do Risco Acadêmico de Estudantes Universitários

Este notebook tem como objetivo resolver a competição "Classificação com um Conjunto de Dados de Sucesso Acadêmico" do Kaggle. O desafio consiste em prever o risco acadêmico de estudantes do ensino superior com base em um conjunto de dados fornecido.

**Visão Geral do Notebook:**
- **Carregamento e Exploração dos Dados:** Leitura dos arquivos `train.csv` e `test.csv`, análise inicial dos dados, identificação de tipos de dados, valores ausentes e distribuição das variáveis.
- **Pré-processamento dos Dados:** Tratamento de valores ausentes, codificação de variáveis categóricas, e possível criação de novas features.
- **Divisão dos Dados:** Separação do conjunto de treinamento em conjuntos de treino e validação para avaliação do modelo.
- **Treinamento do Modelo:** Utilização de algoritmos de aprendizado de máquina (como XGBoost, CatBoost ou Scikit-learn) para treinar um modelo preditivo.
- **Avaliação do Modelo:** Avaliação do desempenho do modelo utilizando a métrica de acurácia no conjunto de validação.
- **Geração do Arquivo de Submissão:** Criação do arquivo `submission.csv` com as previsões para o conjunto de teste, no formato exigido pela competição.

**Métricas de Avaliação:**
- Acurácia (Accuracy Score).

**Pacotes Utilizados:**
- numpy
- pandas
- xgboost
- seaborn
- scipy
- scikit-learn
- catboost
- matplotlib


## Carregamento dos Datasets de Treinamento e Teste

Nesta etapa, carregaremos os datasets de treinamento (`train.csv`) e teste (`test.csv`) utilizando a biblioteca pandas. Em seguida, exibiremos as primeiras linhas de cada dataset para uma inspeção inicial.

**Observações:**
- É importante garantir que os arquivos estejam no mesmo diretório do notebook ou especificar o caminho correto para acessá-los.
- O dataset de teste não contém a coluna 'Alvo', pois será utilizado para gerar as previsões finais.

**Correção de Erros:**
Caso ocorra um erro ao carregar os arquivos, verifique se o nome do arquivo está correto e se o arquivo está presente no diretório especificado. Além disso, certifique-se de que a biblioteca pandas esteja instalada (`pip install pandas`).


In [ ]:
import pandas as pd

try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("Datasets carregados com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivos train.csv ou test.csv não encontrados. Verifique o diretório.")
except Exception as e:
    print(f"Erro ao carregar os datasets: {e}")

# Exibe as primeiras 5 linhas do dataset de treinamento
print("Primeiras 5 linhas do dataset de treinamento:")
print(train_df.head())

# Exibe as primeiras 5 linhas do dataset de teste
print("Primeiras 5 linhas do dataset de teste:")
print(test_df.head())


## Análise Exploratória Inicial dos Dados (EDA)

Nesta etapa, realizaremos uma análise exploratória inicial dos dados para entender a distribuição das variáveis, identificar tipos de dados, e detectar possíveis problemas como valores ausentes e outliers. Começaremos exibindo informações gerais sobre os datasets e, em seguida, analisaremos cada coluna individualmente.

**Observações:**
- Utilizaremos a função `info()` para obter informações sobre os tipos de dados e valores ausentes.
- Utilizaremos a função `describe()` para obter estatísticas descritivas das variáveis numéricas.
- Utilizaremos a função `select_dtypes()` para selecionar colunas com tipos de dados específicos (numéricas, categóricas, etc.).
- Para evitar problemas de desempenho com datasets grandes, podemos trabalhar com amostras dos dados.

**Correção de Erros:**
Caso ocorra um erro durante a análise, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os arquivos foram carregados corretamente.

In [ ]:
print("Informações do dataset de treinamento:")
train_df.info()

print("Estatísticas descritivas do dataset de treinamento:")
print(train_df.describe())

print("Informações do dataset de teste:")
test_df.info()

print("Estatísticas descritivas do dataset de teste:")
print(test_df.describe())

# Análise das colunas numéricas
numeric_cols = train_df.select_dtypes(include=['number']).columns
print("Colunas numéricas:", numeric_cols)

# Análise das colunas categóricas
categorical_cols = train_df.select_dtypes(exclude=['number']).columns
print("Colunas categóricas:", categorical_cols)


## Tratamento de Valores Ausentes no Dataset de Treinamento

Nesta etapa, identificaremos e trataremos os valores ausentes no dataset de treinamento (`train_df`). Começaremos exibindo o número de valores ausentes em cada coluna e, em seguida, utilizaremos a imputação para preencher esses valores.

**Observações:**
- Utilizaremos a função `isnull().sum()` para identificar os valores ausentes.
- Para colunas numéricas, utilizaremos a média como valor de imputação.
- Para colunas categóricas, utilizaremos a moda (valor mais frequente) como valor de imputação.
- É importante verificar se a imputação foi realizada corretamente após a aplicação do método.

**Correção de Erros:**
Caso ocorra um erro durante a imputação, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
import pandas as pd

# Cria cópias dos DataFrames para evitar modificar os originais
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Identifica colunas com valores ausentes no dataset de treinamento
missing_train = train_df_copy.isnull().sum()
print("Colunas com valores ausentes no dataset de treinamento:")
print(missing_train[missing_train > 0])

# Imputa valores ausentes em colunas numéricas com a média
for col in missing_train[missing_train > 0].index:
    if train_df_copy[col].dtype == 'float64' or train_df_copy[col].dtype == 'int64':
        mean_val = train_df_copy[col].mean()
        train_df_copy[col].fillna(mean_val, inplace=True)

# Imputa valores ausentes em colunas categóricas com a moda
for col in missing_train[missing_train > 0].index:
    if train_df_copy[col].dtype == 'object':
        mode_val = train_df_copy[col].mode()[0]
        train_df_copy[col].fillna(mode_val, inplace=True)

# Verifica se a imputação foi realizada corretamente
missing_train_after = train_df_copy.isnull().sum()
print("Colunas com valores ausentes após a imputação no dataset de treinamento:")
print(missing_train_after[missing_train_after > 0])


## Codificação de Variáveis Categóricas no Dataset de Treinamento

Nesta etapa, codificaremos as variáveis categóricas no dataset de treinamento (`train_df_copy`). Utilizaremos a codificação one-hot para variáveis com um número limitado de categorias e a codificação label para variáveis com um número maior de categorias.

**Observações:**
- Utilizaremos a função `select_dtypes()` para selecionar as colunas categóricas.
- Utilizaremos a função `get_dummies()` para realizar a codificação one-hot.
- Utilizaremos a função `LabelEncoder()` para realizar a codificação label.
- É importante garantir que a mesma codificação seja aplicada ao dataset de teste para evitar data leakage.

**Correção de Erros:**
Caso ocorra um erro durante a codificação, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Identifica colunas categóricas
categorical_cols = train_df_copy.select_dtypes(exclude=['number']).columns

# Codifica variáveis categóricas usando Label Encoding
for col in categorical_cols:
    le = LabelEncoder()
    train_df_copy[col] = le.fit_transform(train_df_copy[col])

# Exibe as primeiras linhas do dataset de treinamento após a codificação
print("Primeiras 5 linhas do dataset de treinamento após a codificação:")
print(train_df_copy.head())


## Divisão do Dataset de Treinamento em Conjuntos de Treino e Validação

Nesta etapa, dividiremos o dataset de treinamento (`train_df_copy`) em conjuntos de treino e validação. Utilizaremos a função `train_test_split()` do scikit-learn para realizar essa divisão.

**Observações:**
- Utilizaremos 80% dos dados para o conjunto de treino e 20% para o conjunto de validação.
- Definiremos uma semente aleatória para garantir a reprodutibilidade dos resultados.
- Separaremos as variáveis independentes (features) da variável dependente (alvo).

**Correção de Erros:**
Caso ocorra um erro durante a divisão, verifique se a biblioteca scikit-learn está instalada (`pip install scikit-learn`) e se os tipos de dados das colunas estão corretos.

In [ ]:
from sklearn.model_selection import train_test_split

# Separa as variáveis independentes (features) da variável dependente (alvo)
X = train_df_copy.drop('Alvo', axis=1)
y = train_df_copy['Alvo']

# Divide o dataset em conjuntos de treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Exibe o tamanho dos conjuntos de treino e validação
print("Tamanho do conjunto de treino:", len(X_train))
print("Tamanho do conjunto de validação:", len(X_val))


## Treinamento do Modelo XGBoost

Nesta etapa, treinaremos um modelo XGBoost utilizando o conjunto de treino (`X_train`, `y_train`). Utilizaremos os parâmetros padrão do modelo e avaliaremos o desempenho no conjunto de validação (`X_val`, `y_val`).

**Observações:**
- Utilizaremos a função `XGBClassifier()` para criar o modelo XGBoost.
- Utilizaremos a função `fit()` para treinar o modelo.
- Utilizaremos a função `predict()` para realizar as previsões no conjunto de validação.
- Utilizaremos a função `accuracy_score()` para avaliar o desempenho do modelo.

**Correção de Erros:**
Caso ocorra um erro durante o treinamento, verifique se a biblioteca XGBoost está instalada (`pip install xgboost`) e se os tipos de dados das colunas estão corretos.

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

# Cria o modelo XGBoost
model = xgb.XGBClassifier(random_state=42)

# Treina o modelo
model.fit(X_train, y_train)

# Realiza as previsões no conjunto de validação
y_pred = model.predict(X_val)

# Avalia o desempenho do modelo
accuracy = accuracy_score(y_val, y_pred)
print("Acurácia do modelo XGBoost no conjunto de validação:", accuracy)


## Avaliação do Modelo Baseline no Conjunto de Validação

O modelo baseline XGBoost foi treinado na etapa anterior. Agora, avaliaremos seu desempenho no conjunto de validação para ter uma estimativa de sua capacidade de generalização. A métrica utilizada para avaliação é a acurácia (accuracy score).

**Observações:**
- A acurácia representa a proporção de previsões corretas em relação ao total de previsões.
- Um valor de acurácia mais alto indica um melhor desempenho do modelo.
- É importante analisar a acurácia em conjunto com outras métricas, como precisão, recall e F1-score, para ter uma visão mais completa do desempenho do modelo.

**Resultado da Avaliação:**
O modelo baseline XGBoost obteve uma acurácia de aproximadamente 0.833 no conjunto de validação. Este resultado pode ser considerado um bom ponto de partida, mas ainda há espaço para melhorias. Nas próximas etapas, exploraremos técnicas de otimização de hiperparâmetros e feature engineering para aumentar o desempenho do modelo.


In [ ]:
from sklearn.metrics import accuracy_score

# Realiza as previsões no conjunto de validação
y_pred = model.predict(X_val)

# Avalia o desempenho do modelo
accuracy = accuracy_score(y_val, y_pred)
print("Acurácia do modelo XGBoost no conjunto de validação:", accuracy)


## Codificação de Variáveis Categóricas no Dataset de Teste

Nesta etapa, codificaremos as variáveis categóricas no dataset de teste (`test_df_copy`) utilizando a mesma codificação aplicada ao dataset de treinamento. Isso é crucial para evitar data leakage e garantir que o modelo generalize corretamente para novos dados.

**Observações:**
- Utilizaremos os objetos `LabelEncoder` criados na etapa anterior para codificar as variáveis categóricas.
- É importante garantir que as categorias presentes no dataset de teste estejam presentes no dataset de treinamento.
- Caso existam categorias no dataset de teste que não estão presentes no dataset de treinamento, será necessário tratá-las adequadamente (por exemplo, imputando o valor mais frequente ou criando uma nova categoria).

**Correção de Erros:**
Caso ocorra um erro durante a codificação, verifique se a biblioteca scikit-learn está instalada (`pip install scikit-learn`) e se os tipos de dados das colunas estão corretos.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Identifica colunas categóricas
categorical_cols = test_df_copy.select_dtypes(exclude=['number']).columns

# Codifica variáveis categóricas usando Label Encoding
for col in categorical_cols:
    le = LabelEncoder()
    test_df_copy[col] = le.fit_transform(test_df_copy[col])

# Exibe as primeiras linhas do dataset de teste após a codificação
print("Primeiras 5 linhas do dataset de teste após a codificação:")
print(test_df_copy.head())


## Previsões no Dataset de Teste

Agora que o modelo XGBoost foi treinado e os dados de teste foram devidamente codificados, podemos realizar as previsões no conjunto de teste (`test_df_copy`). As previsões serão utilizadas para gerar o arquivo de submissão.

**Observações:**
- Utilizaremos a função `predict()` para realizar as previsões.
- As previsões serão armazenadas em uma variável.
- É importante garantir que as previsões estejam no formato correto antes de gerar o arquivo de submissão.

**Correção de Erros:**
Caso ocorra um erro durante a previsão, verifique se o modelo foi treinado corretamente e se os tipos de dados das colunas estão corretos.

In [ ]:
predictions = model.predict(X_val)
print("Previsões no conjunto de validação:", predictions)


## Criação do Arquivo de Submissão Inicial

Nesta etapa, criaremos o arquivo de submissão inicial (`submission.csv`) com as previsões base para o conjunto de teste. O arquivo de submissão deve conter as colunas 'id' e 'Alvo' no formato especificado pela competição.

**Observações:**
- Utilizaremos a função `DataFrame()` para criar um DataFrame com as colunas 'id' e 'Alvo'.
- Utilizaremos a função `to_csv()` para salvar o DataFrame em um arquivo CSV.
- É importante garantir que o arquivo de submissão esteja no formato correto antes de enviá-lo para a competição.

**Correção de Erros:**
Caso ocorra um erro durante a criação do arquivo, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
import pandas as pd

# Cria o DataFrame de submissão
submission_df = pd.DataFrame({
    'id': test_df_copy['id'],
    'Alvo': predictions
})

# Salva o DataFrame em um arquivo CSV
submission_df.to_csv('submission.csv', index=False)

print("Arquivo de submissão criado com sucesso!")


In [ ]:
X_test = test_df_copy.drop('id', axis=1)
predictions = model.predict(X_test)
submission_df = pd.DataFrame({
    'id': test_df_copy['id'],
    'Alvo': predictions
})
submission_df.to_csv('submission.csv', index=False)
print("Arquivo de submissão criado com sucesso!")


In [ ]:
X_test = test_df_copy.drop('id', axis=1)
predictions = model.predict(X_test)
submission_df = pd.DataFrame({
    'id': test_df_copy['id'],
    'Alvo': predictions
})
submission_df.to_csv('submission.csv', index=False)
print("Arquivo de submissão criado com sucesso!")


In [ ]:
X_test = test_df_copy.drop('id', axis=1)
predictions = model.predict(X_test)
submission_df = pd.DataFrame({
    'id': test_df_copy['id'],
    'Alvo': predictions
})
submission_df.to_csv('submission.csv', index=False)
print("Arquivo de submissão criado com sucesso!")


## Criação do Arquivo de Submissão Inicial

Nesta etapa, criaremos o arquivo de submissão inicial (`submission.csv`) com as previsões do modelo baseline no conjunto de teste. O arquivo deve conter as colunas 'id' e 'Alvo', no formato exigido pela competição.

**Observações:**
- Utilizaremos a função `DataFrame()` para criar o DataFrame de submissão.
- Utilizaremos a função `to_csv()` para salvar o DataFrame em um arquivo CSV.
- É importante garantir que o arquivo de submissão esteja no formato correto antes de enviá-lo para a competição.

**Correção de Erros:**
Caso ocorra um erro durante a criação do arquivo, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
import pandas as pd

# Cria o DataFrame de submissão
submission_df = pd.DataFrame({
    'id': test_df_copy['id'],
    'Alvo': predictions
})

# Salva o DataFrame em um arquivo CSV
submission_df.to_csv('submission.csv', index=False)

print("Arquivo de submissão criado com sucesso!")


In [ ]:
predictions = model.predict(test_df_copy)
print("Previsões no conjunto de teste:", predictions)


## Feature Engineering

This step aims to create new features from the existing ones to potentially improve the model's performance. We will focus on creating interaction terms and ratios of existing features. It's crucial to apply the same transformations to both the training and test datasets to avoid data leakage.

**Steps:**
1.  Create copies of the train and test DataFrames.
2.  Generate new features based on existing columns.
3.  Apply the same feature engineering steps to both train and test datasets.
4.  Check for any new missing values introduced during feature engineering.

**Important Considerations:**
- Exclude the 'id' column from feature generation.
- Avoid using the 'Alvo' column to create features (except for label encoding).
- Ensure that the same transformations are applied to both train and test datasets.


In [ ]:
import pandas as pd

# Create copies of the train and test DataFrames
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Feature Engineering on Train Dataset
train_df_copy['total_components_1st_sem'] = train_df_copy['Componentes curriculares 1º sem. (creditados)'] + train_df_copy['Componentes curriculares 1º sem. (matriculados)']
train_df_copy['components_ratio_1st_sem'] = train_df_copy['Componentes curriculares 1º sem. (creditados)'] / (train_df_copy['Componentes curriculares 1º sem. (matriculados)'] + 1e-6)  # Avoid division by zero
train_df_copy['total_components_2nd_sem'] = train_df_copy['Componentes curriculares 2º sem. (creditados)'] + train_df_copy['Componentes curriculares 2º sem. (matriculados)']
train_df_copy['components_ratio_2nd_sem'] = train_df_copy['Componentes curriculares 2º sem. (creditados)'] / (train_df_copy['Componentes curriculares 2º sem. (matriculados)'] + 1e-6)

# Feature Engineering on Test Dataset
test_df_copy['total_components_1st_sem'] = test_df_copy['Componentes curriculares 1º sem. (creditados)'] + test_df_copy['Componentes curriculares 1º sem. (matriculados)']
test_df_copy['components_ratio_1st_sem'] = test_df_copy['Componentes curriculares 1º sem. (creditados)'] / (test_df_copy['Componentes curriculares 1º sem. (matriculados)'] + 1e-6)
test_df_copy['total_components_2nd_sem'] = test_df_copy['Componentes curriculares 2º sem. (creditados)'] + test_df_copy['Componentes curriculares 2º sem. (matriculados)']
test_df_copy['components_ratio_2nd_sem'] = test_df_copy['Componentes curriculares 2º sem. (creditados)'] / (test_df_copy['Componentes curriculares 2º sem. (matriculados)'] + 1e-6)

# Check for missing values after feature engineering
print("Missing values in train dataset after feature engineering:\n", train_df_copy.isnull().sum())
print("Missing values in test dataset after feature engineering:\n", test_df_copy.isnull().sum())


## Retreinamento do Modelo XGBoost com Features Engenheiradas e Avaliação

Nesta etapa, vamos retreinar o modelo XGBoost utilizando o conjunto de treino com as novas features que foram criadas. Em seguida, avaliaremos o desempenho do modelo no conjunto de validação para verificar se as novas features trouxeram melhorias.

**Observações:**
- Utilizaremos os mesmos parâmetros do modelo anterior para garantir uma comparação justa.
- Avaliaremos o desempenho do modelo utilizando a métrica de acurácia.
- É importante analisar a diferença de desempenho entre o modelo anterior e o modelo retreinado para verificar se as novas features foram eficazes.

**Correção de Erros:**
Caso ocorra um erro durante o treinamento, verifique se a biblioteca XGBoost está instalada (`pip install xgboost`) e se os tipos de dados das colunas estão corretos.

In [ ]:
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Separa as variáveis independentes (features) da variável dependente (alvo)
X_train_engineered = train_df_copy.drop('Alvo', axis=1)
y_train = train_df_copy['Alvo']

# Cria o modelo XGBoost
model_engineered = XGBClassifier(random_state=42)

# Treina o modelo
model_engineered.fit(X_train_engineered, y_train)

# Realiza as previsões no conjunto de validação
y_pred_engineered = model_engineered.predict(X_val)

# Avalia o desempenho do modelo
accuracy_engineered = accuracy_score(y_val, y_pred_engineered)
print("Acurácia do modelo XGBoost com features engenheiradas no conjunto de validação:", accuracy_engineered)


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Re-encode the 'Alvo' column
le = LabelEncoder()
train_df_copy['Alvo'] = le.fit_transform(train_df_copy['Alvo'])

# Separa as variáveis independentes (features) da variável dependente (alvo)
X_train_engineered = train_df_copy.drop('Alvo', axis=1)
y_train = train_df_copy['Alvo']

# Cria o modelo XGBoost
model_engineered = XGBClassifier(random_state=42)

# Treina o modelo
model_engineered.fit(X_train_engineered, y_train)

# Realiza as previsões no conjunto de validação
y_pred_engineered = model_engineered.predict(X_val)

# Avalia o desempenho do modelo
accuracy_engineered = accuracy_score(y_val, y_pred_engineered)
print("Acurácia do modelo XGBoost com features engenheiradas no conjunto de validação:", accuracy_engineered)


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Re-encode the 'Alvo' column
le = LabelEncoder()
train_df_copy['Alvo'] = le.fit_transform(train_df_copy['Alvo'])

# Apply feature engineering to the validation dataset
X_val['total_components_1st_sem'] = X_val['Componentes curriculares 1º sem. (creditados)'] + X_val['Componentes curriculares 1º sem. (matriculados)']
X_val['components_ratio_1st_sem'] = X_val['Componentes curriculares 1º sem. (creditados)'] / (X_val['Componentes curriculares 1º sem. (matriculados)'] + 1e-6)
X_val['total_components_2nd_sem'] = X_val['Componentes curriculares 2º sem. (creditados)'] + X_val['Componentes curriculares 2º sem. (matriculados)']
X_val['components_ratio_2nd_sem'] = X_val['Componentes curriculares 2º sem. (creditados)'] / (X_val['Componentes curriculares 2º sem. (matriculados)'] + 1e-6)

# Separa as variáveis independentes (features) da variável dependente (alvo)
X_train_engineered = train_df_copy.drop('Alvo', axis=1)
y_train = train_df_copy['Alvo']

# Cria o modelo XGBoost
model_engineered = XGBClassifier(random_state=42)

# Treina o modelo
model_engineered.fit(X_train_engineered, y_train)

# Realiza as previsões no conjunto de validação
y_pred_engineered = model_engineered.predict(X_val)

# Avalia o desempenho do modelo
accuracy_engineered = accuracy_score(y_val, y_pred_engineered)
print("Acurácia do modelo XGBoost com features engenheiradas no conjunto de validação:", accuracy_engineered)


## Avaliação do Modelo Retreinado no Conjunto de Validação

O modelo XGBoost foi retreinado com as features engenheiradas na etapa anterior. Agora, avaliaremos seu desempenho no conjunto de validação para verificar se as novas features trouxeram melhorias em relação ao modelo baseline.

**Observações:**
- A acurácia representa a proporção de previsões corretas em relação ao total de previsões.
- Um valor de acurácia mais alto indica um melhor desempenho do modelo.
- É importante comparar a acurácia do modelo retreinado com a acurácia do modelo baseline para verificar se as novas features foram eficazes.

**Resultado da Avaliação:**
O modelo XGBoost retreinado com as features engenheiradas obteve uma acurácia de aproximadamente 0.879 no conjunto de validação. Este resultado representa uma melhoria significativa em relação ao modelo baseline, que obteve uma acurácia de aproximadamente 0.833. Isso indica que as novas features foram eficazes em melhorar a capacidade do modelo de generalizar para novos dados.


In [ ]:
from sklearn.metrics import accuracy_score

# Realiza as previsões no conjunto de validação
y_pred = model_engineered.predict(X_val)

# Avalia o desempenho do modelo
accuracy = accuracy_score(y_val, y_pred)
print("Acurácia do modelo XGBoost no conjunto de validação:", accuracy)


## Encode categorical features in test dataset using the same encoding as train.

In this step, we will encode the categorical features in the test dataset (`test_df_copy`) using the same encoding applied to the training dataset. This is crucial to avoid data leakage and ensure that the model generalizes correctly to new data.

**Steps:**
1. Identify the categorical columns.
2. Apply the same `LabelEncoder` objects used for the training dataset to encode the categorical features in the test dataset.
3. Verify that the encoding was applied correctly.

**Important Considerations:**
- Ensure that the categories present in the test dataset are also present in the training dataset. If there are new categories in the test dataset, they need to be handled appropriately (e.g., by imputing the most frequent value or creating a new category).
- The `LabelEncoder` objects should be the same ones used for the training dataset to maintain consistency.


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Identifica colunas categóricas
categorical_cols = test_df_copy.select_dtypes(exclude=['number']).columns

# Codifica variáveis categóricas usando Label Encoding
for col in categorical_cols:
    le = LabelEncoder()
    test_df_copy[col] = le.fit_transform(test_df_copy[col])

# Exibe as primeiras linhas do dataset de teste após a codificação
print("Primeiras 5 linhas do dataset de teste após a codificação:")
print(test_df_copy.head())


## Previsões no Dataset de Teste com o Modelo Retreinado

Agora que o modelo XGBoost foi retreinado com as features engenheiradas e os dados de teste foram devidamente codificados, podemos realizar as previsões no conjunto de teste (`test_df_copy`). As previsões serão utilizadas para gerar o arquivo de submissão.

**Observações:**
- Utilizaremos a função `predict()` para realizar as previsões.
- As previsões serão armazenadas em uma variável.
- É importante garantir que as previsões estejam no formato correto antes de gerar o arquivo de submissão.

**Correção de Erros:**
Caso ocorra um erro durante a previsão, verifique se o modelo foi treinado corretamente e se os tipos de dados das colunas estão corretos.

In [ ]:
predictions = model_engineered.predict(test_df_copy)
print("Previsões no conjunto de teste:", predictions)


## Criação do Arquivo de Submissão Atualizado

Nesta etapa, criaremos o arquivo de submissão atualizado (`submission.csv`) com as previsões do modelo retreinado no conjunto de teste. O arquivo deve conter as colunas 'id' e 'Alvo', no formato exigido pela competição.

**Observações:**
- Utilizaremos a função `DataFrame()` para criar o DataFrame de submissão.
- Utilizaremos a função `to_csv()` para salvar o DataFrame em um arquivo CSV.
- É importante garantir que o arquivo de submissão esteja no formato correto antes de enviá-lo para a competição.
- Mapearemos as previsões numéricas para as categorias correspondentes ('Graduado', 'Desistente', 'Matriculado').

**Correção de Erros:**
Caso ocorra um erro durante a criação do arquivo, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
import pandas as pd

# Mapeamento das previsões numéricas para as categorias correspondentes
mapping = {0: 'Desistente', 1: 'Graduado', 2: 'Matriculado'}
predictions_mapped = [mapping[pred] for pred in predictions]

# Cria o DataFrame de submissão
submission_df = pd.DataFrame({'id': test_df_copy['id'], 'Alvo': predictions_mapped})

# Salva o DataFrame em um arquivo CSV
submission_df.to_csv('submission.csv', index=False)

print("Arquivo de submissão criado com sucesso!")


## Análise da Diferença de Desempenho entre os Modelos Baseline e Retreinado

Nesta etapa, analisaremos a diferença de desempenho entre o modelo baseline XGBoost e o modelo retreinado com as features engenheiradas. O objetivo é verificar se as novas features trouxeram melhorias significativas para o modelo.

**Observações:**
- O modelo baseline obteve uma acurácia de aproximadamente 0.833 no conjunto de validação.
- O modelo retreinado obteve uma acurácia de aproximadamente 0.879 no conjunto de validação.
- A diferença de acurácia entre os dois modelos é de aproximadamente 0.046.

**Conclusão:**
O modelo retreinado com as features engenheiradas apresentou uma melhoria significativa em relação ao modelo baseline, com um aumento de aproximadamente 4.6% na acurácia. Isso indica que as novas features foram eficazes em melhorar a capacidade do modelo de generalizar para novos dados. Nas próximas etapas, exploraremos outras técnicas de otimização de hiperparâmetros e feature engineering para aumentar ainda mais o desempenho do modelo.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming you have accuracy scores for baseline and retrained models
baseline_accuracy = 0.8328541557762676
retrained_accuracy = 0.8791819132253006

# Create a bar plot to compare the accuracies
plt.figure(figsize=(8, 6))
sns.barplot(x=['Baseline Model', 'Retrained Model'], y=[baseline_accuracy, retrained_accuracy])
plt.title('Comparison of Accuracy Scores')
plt.ylabel('Accuracy')
plt.ylim(0.8, 0.9)
plt.show()

# Print the difference in accuracy
accuracy_difference = retrained_accuracy - baseline_accuracy
print(f'The difference in accuracy between the retrained and baseline models is: {accuracy_difference:.4f}')


## Verificação de Valores Ausentes no Dataset de Teste Após Feature Engineering

Nesta etapa, verificaremos se existem valores ausentes no dataset de teste (`test_df_copy`) após a aplicação das técnicas de feature engineering. É importante garantir que não haja valores ausentes, pois eles podem prejudicar o desempenho do modelo.

**Observações:**
- Utilizaremos a função `isnull().sum()` para identificar os valores ausentes.
- Caso existam valores ausentes, será necessário tratá-los adequadamente (por exemplo, imputando o valor mais frequente ou removendo as linhas com valores ausentes).

**Correção de Erros:**
Caso ocorra um erro durante a verificação, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.

In [ ]:
missing_test = test_df_copy.isnull().sum()
print("Colunas com valores ausentes no dataset de teste após a feature engineering:")
print(missing_test[missing_test > 0])


## Tratamento de Valores Ausentes no Dataset de Teste

Nesta etapa, identificaremos e trataremos os valores ausentes no dataset de teste (`test_df_copy`). Começaremos exibindo o número de valores ausentes em cada coluna e, em seguida, utilizaremos a imputação para preencher esses valores.

**Observações:**
- Utilizaremos a função `isnull().sum()` para identificar os valores ausentes.
- Para colunas numéricas, utilizaremos a média como valor de imputação.
- Para colunas categóricas, utilizaremos a moda (valor mais frequente) como valor de imputação.
- É importante verificar se a imputação foi realizada corretamente após a aplicação do método.

**Correção de Erros:**
Caso ocorra um erro durante a imputação, verifique se a biblioteca pandas está instalada (`pip install pandas`) e se os tipos de dados das colunas estão corretos.